### Importing database and loading data

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('stores.db')



### Inspecting database

In [2]:
pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

,name
0,customers
1,employees
2,offices
3,orderdetails
4,orders
5,payments
6,productlines
7,products


# Database Overview

The database contains **eight tables** covering customers, employees, sales offices, orders, payments, products, and product categories.

| Table | Description |
|---|---|
| **Customers** | Contains customer information and details. |
| **Employees** | Contains information about all employees. |
| **Offices** | Contains information about the company's sales offices. |
| **Orders** | Contains customers' sales orders. |
| **OrderDetails** | Contains the individual line items associated with each sales order. |
| **Payments** | Contains customers' payment records. |
| **Products** | Contains information about the scale model car products. |
| **ProductLines** | Contains the categories/product lines to which products belong. |

## Database Structure

The database can broadly be divided into three areas:

- **Customers & Employees:** `Customers`, `Employees`, and `Offices`
- **Sales & Payments:** `Orders`, `OrderDetails`, and `Payments`
- **Products & Categories:** `Products` and `ProductLines`

Together, these tables represent a **sales management database for a scale model car business**, covering the complete flow from products and customers to orders, order details, and payments.

## Entity Relationship Diagram (ERD)

The following diagram shows the relationships between the tables in the database.

![Database Entity Relationship Diagram](db.png)

### Displaying first rows of some tables

In [3]:
# Displaying the first five lines from the products table.
pd.read_sql_query("""
 SELECT *
 FROM products
 LIMIT 5;
""", conn)

,productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; deta...,7305,98.58,214.30
2,S10_2016,1996 Moto Guzzi 1100i,Motorcycles,1:10,Highway 66 Mini Classics,"Official Moto Guzzi logos and insignias, saddl...",6625,68.99,118.94
3,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Motorcycles,1:10,Red Start Diecast,"Model features, official Harley Davidson logos...",5582,91.02,193.66
4,S10_4757,1972 Alfa Romeo GTA,Classic Cars,1:10,Motor City Art Classics,Features include: Turnable front wheels; steer...,3252,85.68,136.00


In [4]:
# Counting lines in the products table
pd.read_sql_query("""
    SELECT COUNT(*) AS nb_lines
    FROM products;
""", conn)

,nb_lines
0,110


## Task

Write a SQL query to display the following information:

- Select each **table name** as a string.
- Select the **number of attributes** as an integer (count the number of attributes per table).
- Select the **number of rows** using the `COUNT(*)` function.
- Use the compound operator `UNION ALL` to bind these rows together.


In [5]:
pd.read_sql_query("""
    SELECT 'Customers' AS table_name,
      (SELECT COUNT(*) FROM pragma_table_info('Customers')) AS number_of_attributes,
      (SELECT COUNT(*) FROM Customers) AS number_of_rows
    
    UNION ALL
    
    SELECT 'Products' AS table_name,
      (SELECT COUNT(*) FROM pragma_table_info('Products')) AS number_of_attributes,
      (SELECT COUNT(*) FROM Products) AS number_of_rows
    
    UNION ALL
    
    SELECT 'ProductLines' AS table_name,
      (SELECT COUNT(*) FROM pragma_table_info('ProductLines')) AS number_of_attributes,
      (SELECT COUNT(*) FROM ProductLines) AS number_of_rows
    
    UNION ALL
    
    SELECT 'Orders' AS table_name,
      (SELECT COUNT(*) FROM pragma_table_info('Orders')) AS number_of_attributes,
      (SELECT COUNT(*) FROM Orders) AS number_of_rows
    
    UNION ALL
    
    SELECT 'OrderDetails' AS table_name,
      (SELECT COUNT(*) FROM pragma_table_info('OrderDetails')) AS number_of_attributes,
      (SELECT COUNT(*) FROM OrderDetails) AS number_of_rows
    
    UNION ALL
    
    SELECT 'Payments' AS table_name,
      (SELECT COUNT(*) FROM pragma_table_info('Payments')) AS number_of_attributes,
      (SELECT COUNT(*) FROM Payments) AS number_of_rows
    
    UNION ALL
    
    SELECT 'Employees' AS table_name,
      (SELECT COUNT(*) FROM pragma_table_info('Employees')) AS number_of_attributes,
      (SELECT COUNT(*) FROM Employees) AS number_of_rows
    
    UNION ALL
    
    SELECT 'Offices' AS table_name,
      (SELECT COUNT(*) FROM pragma_table_info('Offices')) AS number_of_attributes,
      (SELECT COUNT(*) FROM Offices) AS number_of_rows;
""", conn)

,table_name,number_of_attributes,number_of_rows
0,Customers,13,122
1,Products,9,110
2,ProductLines,4,7
3,Orders,7,326
4,OrderDetails,5,2996
5,Payments,4,273
6,Employees,8,23
7,Offices,9,7


## Task

### **Question 1 :** Which products should we order more of or less of?

- **Low stock** represents the total quantity ordered for each product divided by the quantity of that product currently in stock:

  **Low stock = SUM(quantityOrdered) / quantityInStock**

  We consider the **ten highest rates** as the top ten products that are almost out of stock or completely out of stock.

- **Product performance** represents the total sales generated by each product:

  **Product performance = SUM(quantityOrdered × priceEach)**

- **Priority products for restocking** are products that combine **high product performance** with a **high low-stock rate**. In other words, these are products that generate significant sales while being close to running out of stock.

### Required Tables

We'll need the following two tables to perform these calculations:

- `Products` — provides product information, including `productCode` and `quantityInStock`.
- `OrderDetails` — provides the ordered quantities and sales prices required to calculate low stock and product performance.

In [6]:
pd.read_sql_query("""
    WITH low_stock_for_product_top_10 AS (
        SELECT 
            p.productCode,
            FLOOR(
                (
                    SELECT SUM(o.quantityOrdered) / p.quantityInStock
                    FROM orderdetails o
                    WHERE o.productCode = p.productCode
                ) * 100
            ) / 100 AS low_stock
        FROM products p
        GROUP BY p.productCode, p.quantityInStock
        ORDER BY low_stock DESC
        LIMIT 10
    ),
    
    product_performance_top_10 AS (
        SELECT
            p.productCode,
            SUM(o.quantityOrdered * o.priceEach) AS product_performance
        FROM products p
        JOIN orderdetails o
            ON o.productCode = p.productCode
        GROUP BY p.productCode
        ORDER BY product_performance DESC
        LIMIT 10
    )
    
    SELECT *
    FROM product_performance_top_10
    WHERE productCode IN (
        SELECT productCode
        FROM low_stock_for_product_top_10
    );
""", conn)

,productCode,product_performance
0,S12_1099,161531.48


### **Question 2 :** How should we match marketing and communication strategies to customer behaviors? 

In [7]:
pd.read_sql_query("""
    SELECT 
        customerNumber,
        SUM(quantityOrdered * (priceEach - buyPrice)) AS profit
    FROM products p
    JOIN orderdetails o
        USING (productCode)
    JOIN orders os
        USING (orderNumber)
    GROUP BY customerNumber
    ORDER BY profit DESC;
""", conn)

,customerNumber,profit
0,141,326519.66
1,124,236769.39
2,151,72370.09
3,114,70311.07
4,119,60875.30
...,...,...
93,489,10868.04
94,103,10063.80
95,473,9532.93
96,198,6586.02


### Finding the VIP and Less Engaged Customers

#### Finding the top five VIP

In [8]:
pd.read_sql_query("""
    WITH profit_per_customer AS(
        SELECT o.customerNumber, SUM(quantityOrdered * (priceEach - buyPrice)) AS profit
          FROM products p
          JOIN orderdetails od
            ON p.productCode = od.productCode
          JOIN orders o
            ON o.orderNumber = od.orderNumber
         GROUP BY o.customerNumber
    )
    
    SELECT contactLastName, contactFirstName, city, country, profit
      FROM customers AS c
      JOIN profit_per_customer AS p
     USING(customerNumber)
    ORDER BY profit DESC
    LIMIT 5;
""", conn)

,contactLastName,contactFirstName,city,country,profit
0,Freyre,Diego,Madrid,Spain,326519.66
1,Nelson,Susan,San Rafael,USA,236769.39
2,Young,Jeff,NYC,USA,72370.09
3,Ferguson,Peter,Melbourne,Australia,70311.07
4,Labrune,Janine,Nantes,France,60875.30


#### Finding the top five least engaged customer

In [9]:
pd.read_sql_query("""
    WITH profit_per_customer AS(
        SELECT o.customerNumber, SUM(quantityOrdered * (priceEach - buyPrice)) AS profit
          FROM products p
          JOIN orderdetails od
            ON p.productCode = od.productCode
          JOIN orders o
            ON o.orderNumber = od.orderNumber
         GROUP BY o.customerNumber
    )
    
    SELECT contactLastName, contactFirstName, city, country, profit
      FROM customers AS c
      JOIN profit_per_customer AS p
     USING(customerNumber)
    ORDER BY profit ASC
    LIMIT 5;
""", conn)

,contactLastName,contactFirstName,city,country,profit
0,Young,Mary,Glendale,USA,2610.87
1,Taylor,Leslie,Brickhaven,USA,6586.02
2,Ricotti,Franco,Milan,Italy,9532.93
3,Schmitt,Carine,Nantes,France,10063.80
4,Smith,Thomas,London,UK,10868.04


### Question 3: How Much Can We Spend on Acquiring New Customers?

Let's Write a query to compute the average of customer profits using the CTE.

In [10]:
pd.read_sql_query("""
    WITH profit_per_customer AS (
        SELECT 
            o.customerNumber,
            SUM(quantityOrdered * (priceEach - buyPrice)) AS profit
        FROM products p
        JOIN orderdetails od
            ON p.productCode = od.productCode
        JOIN orders o
            ON o.orderNumber = od.orderNumber
        GROUP BY o.customerNumber
    )
    
    SELECT AVG(profit) AS average_customer_profit
    FROM profit_per_customer;
""", conn)

,average_customer_profit
0,39039.594388


## 1. Conclusion

This project used SQL to analyze **product performance, stock levels, and customer behavior**. We identified priority products for restocking, VIP and less-engaged customers, and calculated the average customer profit to estimate customer lifetime value.

The analysis also showed a **decline in new customers**, suggesting that investing in customer acquisition could be beneficial. Overall, the results help the company make better decisions about **inventory, marketing, customer loyalty, and acquisition**.

## 2. Project Story

# From Data to Business Decisions

We started by analyzing products to identify those with **high sales but low stock**, helping the company prioritize restocking.

We then analyzed customers to identify **VIP and less-engaged customers**, allowing marketing strategies to be adapted to each group. Finally, we studied customer acquisition and calculated the average customer profit to determine whether investing in new customers would be worthwhile.

**The project shows how SQL can transform raw business data into actionable decisions.**